In [1]:
import torchvision
import matplotlib.pyplot as plt
import os
root_dataset_path = 'dataset/'
if not os.path.exists(root_dataset_path):
    os.makedirs(root_dataset_path)
    train_data = torchvision.datasets.MNIST("./", train=True, transform=torchvision.transforms.ToTensor(), download=True)
    for i,data in enumerate(train_data):
        img = data[0]
        label = data[1]
        son_path = root_dataset_path+str(label)+"/"
        if not os.path.exists(son_path):
            os.makedirs(son_path)
        plt.imsave(son_path+str(len(os.listdir(son_path)))+'.png',img.reshape(28,28), cmap="gray")

In [2]:
from basic import BasicTokenizer
from PIL import Image
import numpy as np
import base64
from bitarray import bitarray
from os.path import exists
import os
import torch
import itertools
import torch.nn as nn
lst = list(itertools.product([0, 1], repeat=8))
bits2token = {}
token2bits = {}
for i in lst:
    token2bits[len(bits2token)] = tuple(i)
    bits2token[tuple(i)] = len(bits2token)

def pre_process(img_bytes):
    '''transform bits to original tokens'''
    tmp=[]
    img_bytes = np.array(img_bytes)
    img_bytes = img_bytes.reshape(-1,8)
    for j in img_bytes:
        tmp.append(bits2token[tuple(j)])
    return tmp
def transform_bits(file):
    ba = bitarray()
    ba.fromfile(file)
    ba = list(ba)
    return ba
tokenizer = BasicTokenizer()
data = []
data2 = []
flag = True
for i in range(1):
    o=0
    path = 'dataset/'+str(i)+'/'
    for j in os.listdir(path):
        if ".ipynb" in j:
            continue
        tmp = os.path.join(path,j)
        f = open(tmp, mode="rb")
        f=transform_bits(f)
        f = pre_process(f)
        data+=f
        data2.append(np.array(f))
tokenizer.train(data, 256 + 30,verbose =True)


merge 1/30: (0, 0) -> 256 ([0, 0]) had 109168 occurrences
merge 2/30: (256, 0) -> 257 ([[0, 0], 0]) had 35783 occurrences
merge 3/30: (116, 112) -> 258 ([116, 112]) had 17898 occurrences
merge 4/30: (108, 111) -> 259 ([108, 111]) had 11982 occurrences
merge 5/30: (105, 98) -> 260 ([105, 98]) had 11936 occurrences
merge 6/30: (15, 97) -> 261 ([15, 97]) had 11874 occurrences
merge 7/30: (116, 108) -> 262 ([116, 108]) had 11864 occurrences
merge 8/30: (257, 28) -> 263 ([[[0, 0], 0], 28]) had 11846 occurrences
merge 9/30: (97, 258) -> 264 ([97, [116, 112]]) had 11846 occurrences
merge 10/30: (264, 259) -> 265 ([[97, [116, 112]], [108, 111]]) had 11846 occurrences
merge 11/30: (265, 262) -> 266 ([[[97, [116, 112]], [108, 111]], [116, 108]]) had 11846 occurrences
merge 12/30: (266, 260) -> 267 ([[[[97, [116, 112]], [108, 111]], [116, 108]], [105, 98]]) had 11846 occurrences
merge 13/30: (256, 261) -> 268 ([[0, 0], [15, 97]]) had 11846 occurrences
merge 14/30: (47, 47) -> 269 ([47, 47]) had 7

In [3]:
from io import BytesIO
def reemovNestings(l):
    lis = []
    for i in l:
        if type(i) == list:
            lis+=reemovNestings(i)
        else:
            lis.append(i)
    return lis
for i in range(len(data2)):
    data2[i] = tokenizer.encode(data2[i])
print("done")




done


In [5]:
import random
block_size = 128
def split(data,split_rate = 0.8):
    l = int(len(data)*(split_rate))
    train = []
    test = []
    for i in range(l):
        train.append(data[i])
    for i in range(l,len(data)):
        test.append(data[i])
    return train, test
def generate_data(data,block_size, repeat_time = 2):
    input = []
    target = []
    for i in range(len(data)):
        for j in range(repeat_time):
            index = random.randint(0,len(data[i])-block_size-2)
            input.append(torch.from_numpy(np.array(data[i][index:index+block_size])))
            target.append(torch.from_numpy(np.array(data[i][index:index+block_size])))
    return input, target
train,test = split(data2,0.8)
input, target  = generate_data(train,block_size,repeat_time=3)


In [4]:
vocab_size = len(tokenizer.vocab)
batch_size = 32
class Sequence(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size,num_layers,batch_size):
        super(Sequence, self).__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.batch_size = batch_size

    def forward(self, x, hidden=None):
        if hidden == None:
            hidden = self.init_hidden(x.shape[0])
        x = self.embed(x)

        out, (h_n, c_n) = self.lstm(x, hidden)
        out = out.contiguous().view(-1, self.hidden_size)
        out = self.linear(out)
        return out
    def init_hidden(self, batch_size):
        h0 = torch.zeros(self.num_layers, self.batch_size, self.hidden_size)

        c0 = torch.zeros(self.num_layers, self.batch_size, self.hidden_size)
        return h0, c0


In [5]:
print(vocab_size)

286


In [7]:
import torch.optim as optim
embedding_dim = 64
hidden_size = 32
num_layers = 1
learning_rate = 0.01
epochs = 50
seq = Sequence(vocab_size,embedding_dim,hidden_size,num_layers,batch_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(seq.parameters(), lr=0.01)
for i in range(4000-batch_size):
    inputs = torch.stack(input[i:i+batch_size]) # fetch words for one seq length  
    targets = torch.stack(target[i:i+batch_size])# shifted by one word from inputs
    outputs = seq(inputs)
    loss = criterion(outputs, targets.reshape(-1))

    seq.zero_grad()
    loss.backward()
        
    #The gradients are clipped in the range [-clip_value, clip_value]. This is to prevent the exploding gradient problem
    optimizer.step()
    if i % 100 == 0:
        print(f'Epoch {i}, Loss: {loss.item()}')

Epoch 0, Loss: 5.6616058349609375
Epoch 100, Loss: 0.022871533408761024
Epoch 200, Loss: 0.00841922964900732
Epoch 300, Loss: 0.004510573577135801
Epoch 400, Loss: 0.003078860929235816
Epoch 500, Loss: 0.002343399915844202
Epoch 600, Loss: 0.0018596070585772395
Epoch 700, Loss: 0.0015087679494172335
Epoch 800, Loss: 0.0012776328949257731
Epoch 900, Loss: 0.0011032144539058208
Epoch 1000, Loss: 0.0009875280084088445
Epoch 1100, Loss: 0.0008705344516783953
Epoch 1200, Loss: 0.0007622644770890474
Epoch 1300, Loss: 0.0006960497121326625
Epoch 1400, Loss: 0.0006405541207641363
Epoch 1500, Loss: 0.0005922173149883747
Epoch 1600, Loss: 0.0005366062978282571
Epoch 1700, Loss: 0.000489750353153795
Epoch 1800, Loss: 0.0004733692039735615
Epoch 1900, Loss: 0.00043321240809746087
Epoch 2000, Loss: 0.00040850244113244116
Epoch 2100, Loss: 0.00037997556501068175
Epoch 2200, Loss: 0.00035922398092225194
Epoch 2300, Loss: 0.0003333386266604066
Epoch 2400, Loss: 0.000316694553475827
Epoch 2500, Loss: 0

In [50]:
from torch.nn import functional as F
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            i = random.randint(0,len(input)-batch_size-1)
            X = torch.stack(input[i:i+batch_size])
            Y = torch.stack(target[i:i+batch_size])
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers

        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)

        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print(vocab_size)

286


In [51]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    i = random.randint(0,len(input)-batch_size-1)

    xb = torch.stack(input[i:i+batch_size])
    yb = torch.stack(target[i:i+batch_size])
    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)

step 0: train loss 5.7972, val loss 5.7994
step 100: train loss 0.3624, val loss 0.3640


KeyboardInterrupt: 